In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

c:\dev\predictive-maintenance-mlops


In [2]:
import numpy as np
import mlflow
from mlflow import MlflowClient
from turbofan.data import RAW_DIR, load_cmapss, add_rul, split_by_engine
from turbofan.features import add_rolling_features, cap_rul, get_feature_columns
from turbofan.quality import check_prediction_range

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()

version = client.get_model_version_by_alias("turbofan-rul", "champion")
model = mlflow.pyfunc.load_model(f"models:/turbofan-rul/{version.version}")
window = int(client.get_run(version.run_id).data.params["window"])

train = cap_rul(add_rul(load_cmapss(RAW_DIR / "train_FD001.txt")))
_, val = split_by_engine(add_rolling_features(train, window=window))

preds = np.asarray(model.predict(val[get_feature_columns()]))
print(preds.min(), preds.max())
check_prediction_range(preds, 0, 125)


c:\dev\predictive-maintenance-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/12 01:28:28 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - jaraco-text (current: uninstalled, required: jaraco-text==4.0.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


1.076767585125385 124.74193994919278


True

In [3]:
import mlflow
from mlflow import MlflowClient
from turbofan.promote import candidate_predictions
from turbofan.quality import check_prediction_range

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()
version = client.get_model_version_by_alias("turbofan-rul", "champion")

preds = candidate_predictions(client, "turbofan-rul", version)
check_prediction_range(preds, 0, 125)
check_prediction_range(preds, 0, 100)

AttributeError: 'ModelVersion' object has no attribute '___'